In [ ]:
# HappyBooking - Step 4: Load batch CSV into Bronze (Delta table)
#
# We read the raw batch CSV that was uploaded to the Lakehouse "Files"
# section, then write it out as a Delta table under "Tables". This is
# our Bronze layer: unprocessed data, kept exactly as it arrived.
#
# multiLine=True is required here because some text fields (e.g.
# review_text) contain embedded newline characters inside quoted
# values. Without this option, Spark's CSV reader splits those quoted
# newlines into extra rows, silently corrupting the row count and
# potentially misaligning columns.

df_batch = spark.read.csv(
    "Files/hotel_raw_batch.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

print(f"Row count: {df_batch.count()}")
print(f"Column count: {len(df_batch.columns)}")


# Write the validated batch DataFrame as a Bronze Delta table.
#
# mode("overwrite") is used here because this is our initial, one-time
# load of the historical batch data. If we re-run this notebook (e.g.
# after fixing an upstream issue), we want a clean replacement rather
# than duplicate rows accumulating on each run.

df_batch.write.format("delta").mode("overwrite").saveAsTable("bronze_hotel_booking_batch")

print("Bronze table 'bronze_hotel_booking_batch' created successfully.")

In [ ]:
result = spark.sql("SELECT COUNT(*) AS row_count FROM bronze_hotel_booking_stream")
result.show()

spark.sql("SELECT * FROM bronze_hotel_booking_stream LIMIT 5").show(truncate=50)

In [8]:
result = spark.sql("SELECT COUNT(*) AS row_count FROM bronze_hotel_booking_stream")
result.show()

StatementMeta(, f3b1dd33-4c55-4fbc-8664-05fcf6b8fec4, 20, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|      342|
+---------+

